# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mayurkharche01/Internship-starter-flyrank/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

This notebook checks whether safe, observable signals contain useful directional information for content refresh prioritization.

The analysis is descriptive and decision-support only. I will not use future-window outcomes, product decision flags, or label-derived fields as inputs.

In [4]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from IPython.display import display

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [5]:
from pathlib import Path

current = Path.cwd().resolve()

print("Current working directory:")
print(current)

# Find the repository root by looking for the data folder
repo_root = None

for candidate in [current] + list(current.parents):
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the FlyRank repository root. "
        "The file data/raw/content_refresh_anonymized.csv was not found."
    )

print("\nRepository root:")
print(repo_root)

Current working directory:
C:\Users\Ashok\Desktop\Mayur\internship-local-project\Internship-starter-flyrank\work\notebooks

Repository root:
C:\Users\Ashok\Desktop\Mayur\internship-local-project\Internship-starter-flyrank


In [6]:
DATA_PATH = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

print("Loading:")
print(DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Loading:
C:\Users\Ashok\Desktop\Mayur\internship-local-project\Internship-starter-flyrank\data\raw\content_refresh_anonymized.csv

Dataset loaded successfully.
Rows: 30000
Columns: 44


In [7]:
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (30000, 44)

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [8]:
print("Available columns:\n")

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

Available columns:

1. content_id
2. client_id
3. search_volume
4. competition
5. competition_level
6. cpc
7. content_type
8. main_intent
9. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct


### Safe-field rule

The following fields are not used as signals because they represent product decisions or outcome/label information:

- `trend_direction`
- `trend_pct`
- `health_score`
- `needs_ctr_fix`
- `is_quick_win`
- `needs_engagement_fix`
- `is_underperformer`
- `is_declining`
- `is_initial_refresh_candidate`

The analysis uses observable search, performance, freshness, and position signals instead.

In [9]:
required_safe_columns = [
    "content_id",
    "impressions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "position_tier"
]

missing_columns = [
    col for col in required_safe_columns
    if col not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

print("All required safe columns are available.")

All required safe columns are available.


## 1. Distributions

I first inspect the main numeric signals before choosing thresholds. This helps identify skew, heavy tails, zeros, and unusual values that could affect a simple rule.

In [10]:
numeric_cols = [
    "impressions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

distribution_summary = df[numeric_cols].describe().T

display(distribution_summary)

,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0


In [11]:
missing_summary = (
    df[numeric_cols]
    .isna()
    .sum()
    .to_frame("missing_n")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_n"] / len(df) * 100
).round(2)

display(missing_summary)

,missing_n,missing_pct
impressions_90d,0,0.0
content_age_days,0,0.0
days_since_last_update,0,0.0
ctr,0,0.0
avg_position,0,0.0


In [12]:
quantiles = (
    df[numeric_cols]
    .quantile([0, 0.25, 0.50, 0.75, 0.90, 0.95, 1.00])
    .T
)

display(quantiles)

,0.00,0.25,0.50,0.75,0.90,0.95,1.00
impressions_90d,1.0,81.0,731.00,3615.25,12136.40,22996.50,517715.0
content_age_days,90.0,132.0,236.00,333.00,463.00,487.00,564.0
days_since_last_update,1.0,20.0,20.00,104.00,104.00,104.00,373.0
ctr,0.0,0.0,0.07,0.29,0.65,1.09,100.0
avg_position,0.0,6.2,10.80,22.30,36.80,48.20,245.0


### Distribution observation

The main numeric signals have different scales and distributions. Some fields show a wider upper tail, so I will use transparent buckets rather than relying only on the mean.

The bucket tables below report `n` so that small groups can be identified before interpreting the direction.

### Signal test #1 — Staleness

Hypothesis: older/staler content may deserve higher refresh-review priority.

I will bucket `days_since_last_update` and compare the observed visibility (`impressions_90d`) across buckets.

This tests whether staleness is directionally useful; it does not claim that age causes performance changes.

In [13]:
staleness_df = df[
    [
        "days_since_last_update",
        "impressions_90d"
    ]
].dropna().copy()

staleness_df["staleness_bucket"] = pd.qcut(
    staleness_df["days_since_last_update"],
    q=4,
    duplicates="drop"
)

staleness_table = (
    staleness_df
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("impressions_90d", "size"),
        mean_impressions=("impressions_90d", "mean"),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

display(staleness_table)

,staleness_bucket,n,mean_impressions,median_impressions
0,"(0.999, 20.0]",15866,3902.104689,363.0
1,"(20.0, 104.0]",13816,6758.867617,1262.0
2,"(104.0, 373.0]",318,2263.147799,30.0


In [14]:
stale_means = staleness_table["median_impressions"].tolist()

print("Median impressions by staleness bucket:")
for bucket, value in zip(
    staleness_table["staleness_bucket"],
    stale_means
):
    print(f"{bucket}: {value:.2f}")

Median impressions by staleness bucket:
(0.999, 20.0]: 363.00
(20.0, 104.0]: 1262.00
(104.0, 373.0]: 30.00


**Verdict: MIXED**

The observed bucket pattern is not consistently directional. Staleness may still provide decision-support value, but it should not dominate the baseline rule.

### Signal test #2 — Visibility / volume

Hypothesis: pages with stronger observed visibility may represent higher-impact opportunities because changes to those pages affect a larger observed traffic base.

I will bucket `impressions_90d` and compare recent sessions across the buckets.

In [15]:
volume_df = df[
    [
        "impressions_90d",
        "sessions_90d"
    ]
].dropna().copy()

volume_df["volume_bucket"] = pd.qcut(
    volume_df["impressions_90d"],
    q=4,
    duplicates="drop"
)

volume_table = (
    volume_df
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("sessions_90d", "size"),
        mean_sessions=("sessions_90d", "mean"),
        median_sessions=("sessions_90d", "median")
    )
    .reset_index()
)

display(volume_table)

,volume_bucket,n,mean_sessions,median_sessions
0,"(0.999, 81.0]",7503,4.161269,2.0
1,"(81.0, 731.0]",7499,8.444193,4.0
2,"(731.0, 3615.25]",7498,23.029341,11.0
3,"(3615.25, 517715.0]",7500,112.637333,54.0


In [16]:
print("Median sessions by impressions bucket:")

for bucket, value in zip(
    volume_table["volume_bucket"],
    volume_table["median_sessions"]
):
    print(f"{bucket}: {value:.2f}")

Median sessions by impressions bucket:
(0.999, 81.0]: 2.00
(81.0, 731.0]: 4.00
(731.0, 3615.25]: 11.00
(3615.25, 517715.0]: 54.00


**Verdict: CONFIRMED**

The observed visibility buckets show a clear directional relationship with sessions. Visibility therefore appears useful as an impact/context signal for a refresh-priority queue.

### Signal test #3 — CTR versus position

Hypothesis: CTR should be interpreted together with search position because the opportunity represented by a CTR value depends on where the page appears.

I will compare observed CTR across position tiers.

In [17]:
ctr_position_df = df[
    [
        "position_tier",
        "ctr",
        "impressions_90d"
    ]
].dropna().copy()

ctr_position_df = ctr_position_df[
    ctr_position_df["impressions_90d"] >= 100
]

ctr_position_table = (
    ctr_position_df
    .groupby("position_tier", observed=False)
    .agg(
        n=("ctr", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

display(ctr_position_table)

,position_tier,n,mean_ctr,median_ctr
0,deep,879,0.055415,0.00
1,page_1,8633,0.354760,0.23
2,page_3_5,6058,0.142359,0.06
3,striking,5903,0.255782,0.15
4,top_3,533,0.334128,0.19


**Verdict: MIXED**

The observed CTR pattern is not perfectly directional across position tiers. CTR remains useful as review context, but position should be considered alongside it.

## 3. The flag-linked test

FlyRank's refresh logic includes a staleness component.

I therefore test whether the observable staleness signal, `days_since_last_update`, shows a useful relationship with visibility.

I do not use the product flag itself. I use the underlying observable signals only.

In [18]:
flag_test = df[
    [
        "days_since_last_update",
        "impressions_90d"
    ]
].dropna().copy()

flag_test["stale_180_plus"] = (
    flag_test["days_since_last_update"] >= 180
)

flag_test["visible_500_plus"] = (
    flag_test["impressions_90d"] >= 500
)

flag_linked_table = (
    flag_test
    .groupby("stale_180_plus")
    .agg(
        n=("impressions_90d", "size"),
        median_impressions=("impressions_90d", "median"),
        mean_impressions=("impressions_90d", "mean")
    )
    .reset_index()
)

display(flag_linked_table)

,stale_180_plus,n,median_impressions,mean_impressions
0,False,29826,742.0,5223.864514
1,True,174,15.5,1172.448276


In [19]:
print("Staleness and visibility counts:")

display(
    pd.crosstab(
        flag_test["stale_180_plus"],
        flag_test["visible_500_plus"],
        margins=True
    )
)

Staleness and visibility counts:


visible_500_plus,False,True,All
stale_180_plus,,,
False,13117,16709,29826
True,157,17,174
All,13274,16726,30000


**Flag-linked verdict: MIXED**

The observable data provides only partial support for the staleness-plus-visibility assumption. I will therefore use the signals cautiously and keep the baseline simple.

## 4. What this means in practice

The signal audit suggests that staleness and visibility provide useful directional context for prioritizing content-review work, while CTR is most useful when interpreted with position.

The baseline should therefore rank pages for human review rather than automatically deciding which pages must be refreshed.

## Self-check

- [ ] Every section above is filled — Markdown thinking and supporting code
- [ ] The notebook runs top to bottom with no errors
- [ ] Three safe signals were tested
- [ ] Each signal has a visible bucket table
- [ ] Each bucket table includes `n`
- [ ] Each signal has a verdict
- [ ] At least one test is linked to a real FlyRank flag
- [ ] No product flags were used as features
- [ ] No future-window values were used
- [ ] No label-derived values were used
- [ ] Claims use careful words: observed, measured, directional, decision-support
- [ ] Notebook committed under `work/notebooks/`